# ML for CS Project: Malware Detection netML Leaderboard Result Reproduction

**Team:** Rhea Madhogarhia, Anushka Agarwal, Rohan Madhogarhia

## Project Summary
The goal of this project is to determine whether we can train a model to detect malware flows in a network. Effective malware detection helps protect systems and users from security threats. We aim to reproduce and potentially improve upon the results of the netML malware detection leaderboard.

## Objectives
1.  **Data Analysis:** Understand the netML dataset and check for class imbalances.
2.  **Model Training:** Experiment with Random Forest, CatBoost, XGBoost, and KNN.
3.  **Evaluation:** Focus on Recall/Sensitivity to minimize false negatives (malware slipping through).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, recall_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Advanced Boosting Libraries
try:
    import xgboost as xgb
    from xgboost import XGBClassifier
except ImportError:
    print("XGBoost not installed. Skipping import.")

try:
    from catboost import CatBoostClassifier
except ImportError:
    print("CatBoost not installed. Skipping import.")

# Set visualization style
sns.set(style="whitegrid")

## 1. Data Loading

Load the dataset from the netML leaderboard. 
*Note: Ensure the data file is in the same directory or update the path below.*

In [ ]:
# TODO: Update path to your actual data file
data_path = 'path/to/netml_dataset.csv'

# df = pd.read_csv(data_path)

# mock dataset 
# REPLACE AND DELTE THIS BLOCK when using real data
data = {
    'feature_1': np.random.rand(1000),
    'feature_2': np.random.rand(1000),
    'feature_3': np.random.rand(1000),
    'label': np.random.choice(['benign', 'malware'], 1000, p=[0.9, 0.1]) # Simulating imbalance
}
df = pd.DataFrame(data)

print("Dataset Shape:", df.shape)
df.head()

## 2. Exploratory Data Analysis (EDA)

### Addressing Class Imbalance
**Feedback Note:** The project feedback highlighted the importance of checking for class imbalance, as malware data is often uneven (mostly benign traffic, very little malware).

We will visualize the distribution of labels here.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='label', data=df)
plt.title('Distribution of Target Classes')
plt.ylabel('Count')
plt.show()

# Calculate exact ratios
class_counts = df['label'].value_counts(normalize=True)
print("Class Ratios:\n", class_counts)

## 3. Preprocessing

We need to encode the target labels (e.g., Benign=0, Malware=1) and scale the features since models like KNN are sensitive to feature magnitude.

In [ ]:
# 1. Handle Missing Values (if any)
df = df.dropna()

# 2. Separate Features and Target
X = df.drop('label', axis=1)
y = df['label']

# 3. Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Classes: {le.classes_}")

# 4. Split Data
# Stratify to maintain the same malware/benign ratio in train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

# 5. Scale Features (Crucial for KNN, helpful for others)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Model Training

We will test the models proposed: **Random Forest, XGBoost, CatBoost, and KNN**.

### Baseline: Random Forest
We use `class_weight='balanced'` to explicitly handle the imbalance found in the EDA step.

In [ ]:
models = {}

# 1. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)
models['Random Forest'] = rf_model

# 2. K-Nearest Neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
models['KNN'] = knn_model

# 3. XGBoost (Gradient Boosting)
# scale_pos_weight is used in XGBoost for imbalance (count(negative) / count(positive))
try:
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_model.fit(X_train_scaled, y_train)
    models['XGBoost'] = xgb_model
except NameError:
    print("XGBoost skipped.")

# 4. CatBoost
try:
    cat_model = CatBoostClassifier(verbose=0, random_state=42)
    cat_model.fit(X_train_scaled, y_train)
    models['CatBoost'] = cat_model
except NameError:
    print("CatBoost skipped.")

## 5. Evaluation

### Metrics Focus: Recall/Sensitivity
from our proposal "It is more prudent and valuable to stop all malware attacks than it is to ensure we don’t get false positives."* 

Therefore, we look closely at **Recall** for the Malware class.

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    
    results.append({'Model': name, 'Accuracy': acc, 'Recall (Sensitivity)': rec})
    
    print(f"--- {name} Results ---")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    # Confusion Matrix Visualization
    plt.figure(figsize=(4, 3))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', 
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Summary Table
results_df = pd.DataFrame(results)
results_df.sort_values(by='Recall (Sensitivity)', ascending=False)

## 6. Next Steps & Conclusion

- **Feature Importance:** Analyze which network features (from the netML dataset) contributed most to malware detection.
- **Hyperparameter Tuning:** Use GridSearch or RandomizedSearch to improve the best-performing model.
- **Ensembling:** If individual models perform well, consider the weighted ensemble approach mentioned in the proposal (AutoGluon or manual voting).